# 02 Schema Validation

P0 checks for required columns and P1 checks for schema drift.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime
import json

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
    CRITICAL_COLUMNS,
    PRIMARY_KEYS,
    FOREIGN_KEYS,
    RANGE_RULES,
    THRESHOLDS,
    TECHNICAL_KEY_COLUMNS,
    DOMAIN_REVIEW_COLUMNS,
    STRUCTURAL_OPTIONAL_COLUMNS,
    ALLOWED_NULL_SCENARIOS,
    VALIDATION_SEVERITY,
    RANGE_SEVERITY_OVERRIDES,
)
from file_utils import build_file_inventory, endpoint_files, endpoint_files, iter_csv_endpoint
from validation_utils import endpoint_columns, null_profile, duplicate_count, range_violations

NOTEBOOK_NAME = "02_schema_validation"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "PASS"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)


BRONZE VALIDATION - 02_schema_validation
Start time: 2026-06-01 17:59:19.454498
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [ ]:
inventory_path = ROOT / "eda" / "bronze" / "outputs" / "tables" / "01_file_integrity" / "file_integrity.csv"
if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
else:
    inventory = build_file_inventory(RAW_DATA_PATH, EXPECTED_ENDPOINTS)

records = []
for endpoint in EXPECTED_ENDPOINTS:
    endpoint_rows = inventory[inventory["endpoint"].eq(endpoint)]
    observed = []
    for value in endpoint_rows["columns"].dropna().astype(str):
        for column in value.split("|"):
            if column and column not in observed:
                observed.append(column)
    required = CRITICAL_COLUMNS.get(endpoint, [])
    missing = [column for column in required if column not in observed]
    extra = [column for column in observed if column not in required]
    records.append({
        "endpoint": endpoint,
        "required_columns": "|".join(required),
        "observed_columns": "|".join(observed),
        "missing_required_count": len(missing),
        "missing_required_columns": "|".join(missing),
        "extra_columns_count": len(extra),
        "status": "FAIL" if missing else "PASS",
    })
schema_df = pd.DataFrame(records)
schema_df.to_csv(OUTPUT_TABLES / "schema_validation.csv", index=False)
display(schema_df)

,endpoint,required_columns,observed_columns,missing_required_count,missing_required_columns,extra_columns_count,status
0,meetings,meeting_key|meeting_name|year,meeting_key|meeting_name|meeting_official_name...,0,,15,PASS
1,sessions,session_key|meeting_key|session_name|date_start,session_key|session_type|session_name|date_sta...,0,,11,PASS
2,drivers,session_key|driver_number|full_name,meeting_key|session_key|driver_number|broadcas...,0,,9,PASS
3,session_result,session_key|driver_number|position,position|driver_number|number_of_laps|dnf|dns|...,0,,8,PASS
4,laps,session_key|driver_number|lap_number|lap_duration,meeting_key|session_key|driver_number|lap_numb...,0,,12,PASS
5,weather,session_key|date|track_temperature|air_tempera...,date|session_key|air_temperature|track_tempera...,0,,6,PASS
6,stints,,meeting_key|session_key|stint_number|driver_nu...,0,,8,PASS
7,starting_grid,,position|driver_number|lap_duration|meeting_ke...,0,,5,PASS
8,intervals,,date|session_key|gap_to_leader|meeting_key|dri...,0,,6,PASS
9,position,,date|session_key|driver_number|position|meetin...,0,,5,PASS


In [ ]:
plot_df = schema_df[["endpoint", "missing_required_count", "extra_columns_count"]].melt(id_vars="endpoint", var_name="metric", value_name="count")
fig = px.bar(plot_df, x="endpoint", y="count", color="metric", barmode="group", title="Schema Validation: Missing Required and Extra Columns")
fig.update_xaxes(tickangle=35)
fig.write_html(OUTPUT_CHARTS / "schema_validation.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "schema_validation.png")
except Exception:
    pass
fig.show()

In [ ]:
failed = schema_df[schema_df["status"] == "FAIL"]
report = {"notebook": NOTEBOOK_NAME, "timestamp": datetime.now().isoformat(), "p0_status": "PASS" if failed.empty else "FAIL", "results": schema_df.to_dict("records")}
write_report("schema_validation", report)
write_insight(
    "02_schema_insights.md",
    "Schema Validation Insights",
    f"Validated required schemas for {len(schema_df)} endpoints.",
    [f"Endpoints with missing required columns: {len(failed)}"],
    [f"{row.endpoint}: missing {row.missing_required_columns}" for row in failed.itertuples()],
    ["Update schema contracts or adjust crawler/cleaning logic for missing required columns."],
    ["Run 03_pk_fk_checks.ipynb"],
)
print(report["p0_status"])

PASS
